# Computer Vision — CCA 3: Practical Coding Assignment
## Units 3, 4 and 5 (Student ID: 1272251039)

This notebook implements the four tasks of the CCA 3 practical coding assignment:

| Section | Task | Techniques |
|---|---|---|
| 1 | **Image Processing** | Noise synthesis, denoising, PSNR/SSIM, Otsu, morphology, Canny, watershed |
| 2 | **Feature Extraction & Matching** | HOG, SIFT, Brute-Force matching, invariance analysis |
| 3 | **Object Detection** | Viola–Jones faces, YOLOv8 Nano on image + video frame |
| 4 | **Video Analysis** | Frame extraction, frame differencing, MOG2, Lucas–Kanade optical flow |

All figures are saved automatically to `3. Outputs/Task N/`.

In [ ]:
# 0.1 — Required libraries (run once per session)
# A fresh environment is auto-detected; already-importable packages are skipped.
import importlib.util, sys

REQUIRED = ["numpy", "matplotlib", "cv2", "skimage", "sklearn", "ultralytics"]
missing = [m for m in REQUIRED if importlib.util.find_spec(m) is None]

if missing:
    print(f"Missing: {missing} -> run the pip cell below (0.2), then re-run cells from 1.1.")
else:
    print("All required packages are available — skipping install.")

In [ ]:
# 0.2 — Package install command (I-Python magic; works in Colab and Jupyter).
# Kept as its own cell so it can be run manually in any environment.
!pip install -q numpy matplotlib "opencv-python<5" scikit-image scikit-learn ultralytics
print("pip install step finished.")

## 1 — Configuration

A **single configuration cell** defines every path used below. The notebook auto-detects
whether it is running on Google Colab or locally.

**Colab:** upload the three input files to `/content` (or mount Drive and set `INPUT_DIR`).
**Local:** the notebook lives in `<repo>/2. Source Code/`, so the repository root is one level up.

In [ ]:
# 1.1 — Paths and environment
import pathlib, sys

def _is_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False

ENVIRONMENT = "auto"          # "auto" | "colab" | "local"   (override by hand if needed)
if ENVIRONMENT == "auto":
    ENVIRONMENT = "colab" if _is_colab() else "local"

if ENVIRONMENT == "colab":
    # Input files are uploaded to /content (step 4 of the README instructions).
    REPO_ROOT = pathlib.Path("/content")
    INPUT_DIR = pathlib.Path("/content")
    # Optional: mount Drive instead of uploading, e.g.
    # from google.colab import drive; drive.mount('/content/drive')
    # INPUT_DIR = pathlib.Path("/content/drive/MyDrive/CV-CCA-3/1. Original Inputs")
else:                            # local Jupyter: notebook sits in <repo>/2. Source Code/
    REPO_ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "2. Source Code" else pathlib.Path.cwd()
    INPUT_DIR = REPO_ROOT / "1. Original Inputs"

OUTPUT_DIR = REPO_ROOT / "3. Outputs"
ORIGINAL_IMG = INPUT_DIR / "original.jpg"
SECOND_IMG   = INPUT_DIR / "second.jpg"
VIDEO        = INPUT_DIR / "original_video.mp4"

SEED = 1272251039               # reproducibility
print("ENVIRONMENT :", ENVIRONMENT)
print("INPUT_DIR   :", INPUT_DIR)
print("OUTPUT_DIR  :", OUTPUT_DIR)

In [ ]:
# 1.2 — Imports and reproducibility
import os, random, math
import numpy as np
import cv2
import matplotlib
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans                            # used in Task 1 (colour quantisation)
from skimage.metrics import structural_similarity as ssim    # used in Task 1 (SSIM)

random.seed(SEED)
np.random.seed(SEED)

matplotlib.rcParams["figure.dpi"] = 110

print("numpy      :", np.__version__)
print("opencv     :", cv2.__version__)
print("matplotlib :", matplotlib.__version__)

In [ ]:
# 1.3 — Shared helpers (display / save / banner) used by every task

def bgr2rgb(img):
    '''BGR -> RGB for Matplotlib display (pass-through for 2-D grey images).'''
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB) if img.ndim == 3 and img.shape[2] == 3 else img

def show_image(img, title="", cmap=None, size=None, ax=None):
    '''Show one image on the given (or a new) axis, converting BGR to RGB.'''
    tmp = bgr2rgb(img)
    if cmap is None and tmp.ndim == 2:
        cmap = "gray"
    if ax is None:
        fig, ax = plt.subplots(figsize=size)
    ax.imshow(tmp, cmap=cmap)
    ax.set_title(title)
    ax.axis("off")
    return ax

def show_grid(images, titles, cols=3, width_per=3.3, save_name=None, task=None):
    '''Grid of images with titles; optionally saves the whole figure to Outputs.'''
    rows = math.ceil(len(images) / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(width_per * cols, width_per * rows))
    axes = np.atleast_1d(axes).ravel()
    for ax in axes:
        ax.axis("off")
    for ax, img, title in zip(axes, images, titles):
        show_image(img, title=title, ax=ax)
    fig.tight_layout()
    if save_name is not None:
        path = OUTPUT_DIR / ("Task " + str(task))
        path.mkdir(parents=True, exist_ok=True)
        fig.savefig(path / save_name, bbox_inches="tight")
        print("saved ->", path / save_name)
    plt.show()

def save_output(task, filename, image):
    '''Save a BGR or grey image into 3. Outputs/Task <task>/<filename>.'''
    out = OUTPUT_DIR / ("Task " + str(task))
    out.mkdir(parents=True, exist_ok=True)
    path = out / filename
    ok = cv2.imwrite(str(path), image)
    print(("saved -> " if ok else "FAILED  -> ") + str(path))
    return path

def banner(text):
    print("")
    print("=" * 72)
    print(" " + text)
    print("=" * 72)

In [ ]:
# 1.4 — Dataset verification (step 5 of the README execution instructions)
# Confirms the three required input files exist and reports their basic properties.

def _exists(p):
    return p.is_file()

for name, path in [("original.jpg", ORIGINAL_IMG), ("second.jpg", SECOND_IMG), ("original_video.mp4", VIDEO)]:
    if _exists(path):
        if name.endswith(".mp4"):
            cap = cv2.VideoCapture(str(path))
            frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
            fps = cap.get(cv2.CAP_PROP_FPS)
            print(f"{name:<20} OK  | {frames} frames @ {fps:.2f} fps")
            cap.release()
        else:
            img = cv2.imread(str(path))
            print(f"{name:<20} OK  | shape = {img.shape}")
    else:
        raise FileNotFoundError(
            f"{name} not found at: {path}\n"
            "Place all input files inside the input folder shown above "
            "(step 4 of the README instructions) and re-run this cell.")

---
# Task 1 — Image Processing

## 1.1 — Load image and overview

The original image is loaded once and reused by every sub-section of Task 1.
Displayed below: colour original and its grayscale version (grayscale drives thresholding,
noise handling and edge detection).

In [ ]:
# 1.1.1 — Load original image
original_bgr = cv2.imread(str(ORIGINAL_IMG))
assert original_bgr is not None, ("Could not read: " + str(ORIGINAL_IMG))
original_img = original_bgr.copy()   # kept in OpenCV BGR; helpers convert at display
gray = cv2.cvtColor(original_bgr, cv2.COLOR_BGR2GRAY)
H, W = gray.shape
print("Loaded", ORIGINAL_IMG.name, "-", original_bgr.shape)

show_grid(
    [original_img, gray],
    ["Original (colour)", "Grayscale"],
    cols=2, width_per=4.0,
    save_name="1_original_and_gray.png", task=1,
)

## 1.2 — Noise generation

Two classic degradation models are synthesised on the original image:

- **Gaussian noise** — additive zero-mean noise sampled from a normal distribution
  (models sensor / low-light noise, roughly uniform everywhere).
- **Salt-and-Pepper noise** — impulse noise: a fraction of pixels are forced to pure
  white (salt) or pure black (pepper) (models data-dropout / transmission errors).

In [ ]:
# 1.2.1 — Noise synthesis functions
def add_gaussian_noise(img, mean=0.0, sigma=20.0):
    '''Additive Gaussian noise = image + N(mean, sigma) per-pixel.'''
    noise = np.random.normal(mean, sigma, img.shape)
    noisy = img.astype(np.float64) + noise
    return np.clip(noisy, 0, 255).astype(np.uint8)

def add_salt_pepper_noise(img, amount=0.05):
    '''Impulse noise: 'amount' fraction of pixels become salt (255) or pepper (0).'''
    out = img.copy()
    n = int(np.ceil(amount * img.shape[0] * img.shape[1]))
    # salt
    ys, xs = np.random.randint(0, img.shape[0], n), np.random.randint(0, img.shape[1], n)
    for y, x in zip(ys, xs):
        out[y, x] = 255 if out.ndim == 2 else [255, 255, 255]
    # pepper
    ys, xs = np.random.randint(0, img.shape[0], n), np.random.randint(0, img.shape[1], n)
    for y, x in zip(ys, xs):
        out[y, x] = 0 if out.ndim == 2 else [0, 0, 0]
    return out

gauss_noisy = add_gaussian_noise(original_img, sigma=20.0)
sp_noisy    = add_salt_pepper_noise(original_img, amount=0.05)

show_grid(
    [original_img, gauss_noisy, sp_noisy],
    ["Original", "Gaussian noise (sigma=20)", "Salt & Pepper noise (5%)"],
    cols=3, width_per=4.0,
    save_name="2_noise_types.png", task=1,
)

## 1.3 — Denoising with Median and Gaussian filters

- **Median filter** replaces each pixel with the median of its neighbourhood — it removes
  impulse (salt & pepper) noise almost completely, because outlier values are never the median.
- **Gaussian filter** is a weighted average — it suppresses additive Gaussian noise by
  local averaging but also blurs edges.

For fairness, both filters are applied to *both* noisy variants:

In [ ]:
# 1.3.1 — Apply both filters to both noisy images
K_BOX, K_MED = (5, 5), 5

den_results = {
    "orig":                     original_img,
    "gauss noisy":              gauss_noisy,
    "gauss noisy | median":     cv2.medianBlur(gauss_noisy, K_MED),
    "gauss noisy | gaussian":   cv2.GaussianBlur(gauss_noisy, K_BOX, 0),
    "sp noisy":                 sp_noisy,
    "sp noisy | median":        cv2.medianBlur(sp_noisy, K_MED),
    "sp noisy | gaussian":      cv2.GaussianBlur(sp_noisy, K_BOX, 0),
}
show_grid(
    list(den_results.values()), list(den_results.keys()),
    cols=4, width_per=3.4,
    save_name="3_denoising.png", task=1,
)
# keep the primary denoised results for the quality table below
med_on_gauss, gau_on_gauss = den_results["gauss noisy | median"], den_results["gauss noisy | gaussian"]
med_on_sp,    gau_on_sp    = den_results["sp noisy | median"],    den_results["sp noisy | gaussian"]

## 1.4 — Image quality evaluation (PSNR and SSIM)

Quality is measured against the clean original:

- **PSNR (dB)** — pixel fidelity; derived from the mean squared error.
  Typical: > 30 dB = good reconstruction. Infinite when images are identical.
- **SSIM** — perceptual structural similarity in [-1, 1]; compares luminance, contrast
  and structure. 1.0 = identical structure.

Computed on the grayscale version (SSIM is defined per-channel otherwise).

In [ ]:
# 1.4.1 — PSNR / SSIM table for all degradation / restoration combinations
def psnr_db(a, b):
    '''Peak Signal-to-Noise Ratio in dB (uses OpenCV util).'''
    return float(cv2.PSNR(a, b))

def ssim_score(a, b):
    '''SSIM between grayscale versions of a and b (uint8, data_range=255).'''
    ga, gb = cv2.cvtColor(a, cv2.COLOR_BGR2GRAY), cv2.cvtColor(b, cv2.COLOR_BGR2GRAY)
    return float(ssim(ga, gb, data_range=255))

rows = [
    ("Original",                        original_img),
    ("Gaussian noise",                  gauss_noisy),
    ("Gaussian noise -> median filter", med_on_gauss),
    ("Gaussian noise -> gaussian flt",  gau_on_gauss),
    ("Salt&Pepper noise",               sp_noisy),
    ("Salt&Pepper -> median filter",    med_on_sp),
    ("Salt&Pepper -> gaussian filter",  gau_on_sp),
]

ncols = max(len(r[0]) for r in rows) + 4
print("Combination".ljust(ncols), "PSNR (dB)".rjust(10), "SSIM".rjust(8))
print("-" * (ncols + 19))
for label, img in rows:
    p, s = psnr_db(original_img, img), ssim_score(original_img, img)
    print(label.ljust(ncols), f"{p:10.2f}", f"{s:8.4f}")

# qualitative side-by-side: best denoising results vs originals
show_grid(
    [gauss_noisy, gau_on_gauss, sp_noisy, med_on_sp],
    ["Gaussian noisy", "Restored (gaussian flt)", "S&P noisy", "Restored (median flt)"],
    cols=4, width_per=3.6,
    save_name="4_psnr_ssim_compared.png", task=1,
)

## 1.5 — Colour quantisation with KMeans (scikit-learn)

KMeans clusters the pixel colours into K dominant colours — this both produces a poster-like
segmented image and demonstrates the required scikit-learn usage. Increasing K approximates
the original; decreasing K abstracts it.

In [ ]:
# 1.5.1 — KMeans colour quantisation (K = 2, 4, 8)
Z = (original_img.reshape(-1, 3).astype(np.float32)) / 255.0
quantisers = {}
for K in (2, 4, 8):
    km = KMeans(n_clusters=K, n_init="auto", random_state=SEED).fit(Z)
    quant = km.cluster_centers_[km.labels_].reshape(original_img.shape).astype(np.float32)
    quantisers[K] = np.clip(quant * 255, 0, 255).astype(np.uint8)

show_grid(
    [original_img, quantisers[2], quantisers[4], quantisers[8]],
    ["Original", "KMeans K=2", "KMeans K=4", "KMeans K=8"],
    cols=4, width_per=3.4,
    save_name="5_kmeans_quantisation.png", task=1,
)

## 1.6 — Otsu thresholding

Otsu automatically selects the threshold value that **minimises intra-class variance**
(equivalently, maximises inter-class variance) of the grayscale histogram. The red line on
the histogram marks the automatically chosen cut point; the binary image separates the
image into two classes.

In [ ]:
# 1.6.1 — Otsu threshold + histogram visualisation
otsu_val, otsu_bin = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
print("Otsu threshold =", otsu_val)

fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
show_image(gray, title="Grayscale", ax=axes[0])
axes[1].hist(gray.ravel(), bins=256, range=(0, 256), color="steelblue")
axes[1].axvline(otsu_val, color="red", lw=2)
axes[1].set_title(f"Grayscale histogram (Otsu = {otsu_val:.0f})")
show_image(otsu_bin, title="Otsu binary result", ax=axes[2])
fig.tight_layout()
out_dir = OUTPUT_DIR / "Task 1"
out_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(out_dir / "6_otsu_threshold.png", bbox_inches="tight")
print("saved ->", out_dir / "6_otsu_threshold.png")
plt.show()

## 1.7 — Morphological operations

Applied to the Otsu binary image with a 5x5 structuring element:

| Operation | Effect |
|---|---|
| Erosion | shrinks bright regions, removes small specks |
| Dilation | grows bright regions, fills small holes |
| Opening | erosion then dilation — removes small objects |
| Closing | dilation then erosion — closes small gaps |
| Gradient | difference between dilation and erosion — object outline |

In [ ]:
# 1.7.1 — Morphological transformations
kernel = np.ones((5, 5), np.uint8)
morph_out = {
    "Otsu binary": otsu_bin,
    "Erosion":     cv2.erode(otsu_bin, kernel, iterations=1),
    "Dilation":    cv2.dilate(otsu_bin, kernel, iterations=1),
    "Opening":     cv2.morphologyEx(otsu_bin, cv2.MORPH_OPEN, kernel),
    "Closing":     cv2.morphologyEx(otsu_bin, cv2.MORPH_CLOSE, kernel),
    "Gradient":    cv2.morphologyEx(otsu_bin, cv2.MORPH_GRADIENT, kernel),
}
show_grid(
    list(morph_out.values()), list(morph_out.keys()),
    cols=3, width_per=3.6,
    save_name="7_morphological_ops.png", task=1,
)
morph_open = morph_out["Opening"]

## 1.8 — Canny edge detection

Canny = Gaussian smoothing -> gradient magnitude -> non-maximum suppression ->
double-threshold hysteresis. Comparing edges found on the raw grayscale vs a
pre-blurred version shows how smoothing suppresses noise-induced spurious edges.

In [ ]:
# 1.8.1 — Canny edges (raw vs pre-smoothed)
edges_raw    = cv2.Canny(gray, 100, 200)
gray_blurred = cv2.GaussianBlur(gray, (5, 5), 0)
edges_smooth = cv2.Canny(gray_blurred, 100, 200)

show_grid(
    [gray, edges_raw, edges_smooth],
    ["Grayscale", "Canny (raw)", "Canny (after Gaussian blur)"],
    cols=3, width_per=4.0,
    save_name="8_canny_edges.png", task=1,
)

## 1.9 — Watershed segmentation

Watershed treats intensity (or distance-transform peaks) as a topographic surface and
"floods" basins from marked seeds. Marker-based pipeline:

1. Otsu threshold -> morphological opening (denoise the mask)
2. **Sure background** = dilated opened mask
3. **Sure foreground** = high values of the distance transform (region centres)
4. `cv2.connectedComponents` labels seeds; watershed floods from them; boundary pixels
   are marked **-1** and drawn in red on the original image.

In [ ]:
# 1.9.1 — Marker-based watershed segmentation
# Adaptive polarity: choose the threshold direction based on overall brightness so that
# the mask marks "objects" as white against darker surroundings (or vice versa).
flag = (cv2.THRESH_BINARY if np.mean(gray) < 127 else cv2.THRESH_BINARY_INV) + cv2.THRESH_OTSU
_, thr = cv2.threshold(gray, 0, 255, flag)
thr = cv2.morphologyEx(thr, cv2.MORPH_OPEN, kernel, iterations=2)
if thr.max() == 0:
    thr = morph_out["Opening"]

sure_bg = cv2.dilate(thr, kernel, iterations=3)
dist = cv2.distanceTransform(thr, cv2.DIST_L2, 5)
_, sure_fg = cv2.threshold(dist, 0.7 * dist.max() if dist.max() > 0 else 0, 255, 0)
sure_fg = np.uint8(sure_fg)
unknown = cv2.subtract(sure_bg, sure_fg)

n_labels, markers = cv2.connectedComponents(sure_fg)
markers = markers + 1                       # background becomes 1, seeds 2..n
markers[unknown == 255] = 0                 # unlabelled zone for watershed
markers = cv2.watershed(original_bgr, markers.astype(np.int32))

watershed_vis = original_bgr.copy()
watershed_vis[markers == -1] = [0, 0, 255]  # red segmentation boundaries (BGR)
print(f"watershed finished: {n_labels} seed regions")

show_grid(
    [original_img, thr, sure_fg, watershed_vis],
    ["Original", "Foreground mask", "Sure foreground (dist transform)", "Watershed result (red boundaries)"],
    cols=4, width_per=3.6,
    save_name="9_watershed_segmentation.png", task=1,
)

### Task 1 — Results summary

- Median filter is the clear choice for salt & pepper noise; Gaussian filter only
  attenuates additive Gaussian noise at the cost of mild blur.
- PSNR/SSIM quantify this: denoised results score higher than their noisy inputs.
- KMeans quantisation with K=8 stays closest to the original appearance.
- Otsu + morphology give robust binary masks; watershed turns them into labelled regions.
- All figures are saved under `3. Outputs/Task 1/`.

## Task 2 — Feature Extraction and Matching

Pipeline in this section:

1. **HOG** descriptor extraction and visualisation (plus an OpenCV person-detection demo).
2. **SIFT** keypoint detection on both input images.
3. **Brute-Force matching** with the Lowe ratio test between the two images.
4. **Invariance analysis** — matching after rotation and resizing of the original.
5. **Quantitative comparison** of all matching configurations.

## 2.1 — Histogram of Oriented Gradients (HOG)

HOG describes local appearance by counting gradient orientations in dense 8x8 pixel cells,
contrast-normalised over overlapping 2x2-cell blocks. skimage returns:

- a normalised **feature vector** — suitable input to a classifier such as an SVM;
- a **visualisation** image showing the dominant orientation per cell.

In [ ]:
# 2.1.1 — HOG descriptors with skimage
from skimage.feature import hog as skimage_hog

fd, hog_vis = skimage_hog(
    gray,
    orientations=9,
    pixels_per_cell=(8, 8),
    cells_per_block=(2, 2),
    block_norm="L2-Hys",
    visualize=True,
)
print(f"image {W}x{H} -> HOG feature vector length = {fd.size}")
print("feature vector: min {:.3f} | max {:.3f} | mean {:.4f}".format(fd.min(), fd.max(), fd.mean()))

show_grid(
    [gray, hog_vis],
    ["Grayscale input", "HOG visualisation (9 orientations, 8x8 cells)"],
    cols=2, width_per=5.0,
    save_name="10_hog_visualisation.png", task=2,
)

In [ ]:
# 2.1.2 — Optional: OpenCV built-in HOG + linear-SVM person detector
hog_detector = cv2.HOGDescriptor()
hog_detector.setSVMDetector(cv2.HOGDescriptor_getDefaultPeopleDetector())
locations, weights = hog_detector.detectMultiScale(
    original_bgr, winStride=(8, 8), padding=(4, 4), scale=1.05)

person_img = original_bgr.copy()
for (x, y, wB, hB) in locations:
    cv2.rectangle(person_img, (x, y), (x + wB, y + hB), (0, 255, 0), 2)
print(f"HOG person detector found {len(locations)} candidate region(s)")
show_image(person_img, "OpenCV HOG person detection", size=(6, 5))
save_output(2, "11_hog_person_detection.png", person_img)

## 2.2 — Scale-Invariant Feature Transform (SIFT)

SIFT locates scale-space extrema and describes each keypoint with a 128-D gradient
histogram — largely invariant to scale, rotation and illumination shifts. Rich-keypoint
rendition shows the **size and orientation** of every detected point.

In [ ]:
# 2.2.1 — Load second image, detect SIFT keypoints on both
second_bgr = cv2.imread(str(SECOND_IMG))
assert second_bgr is not None, ("Could not read: " + str(SECOND_IMG))
second_img = second_bgr.copy()   # BGR, same convention as original_img
gray2 = cv2.cvtColor(second_bgr, cv2.COLOR_BGR2GRAY)

sift = cv2.SIFT_create()
kp1, des1 = sift.detectAndCompute(gray, None)    # original image
kp2, des2 = sift.detectAndCompute(gray2, None)   # second image
print(f"SIFT keypoints: original = {len(kp1)}, second = {len(kp2)}")

img_kp1 = cv2.drawKeypoints(original_img, kp1, None, flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)
img_kp2 = cv2.drawKeypoints(second_img, kp2, None, flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)
show_grid(
    [img_kp1, img_kp2],
    ["SIFT keypoints - original (%d)" % len(kp1), "SIFT keypoints - second (%d)" % len(kp2)],
    cols=2, width_per=5.0,
    save_name="12_sift_keypoints.png", task=2,
)

## 2.3 — Brute-Force matching (Lowe ratio test)

For every SIFT descriptor of the original, `knnMatch` (k=2) finds its two nearest
neighbours in the second image. The **Lowe ratio test** (0.75) keeps a match only when it
is much closer than the runner-up, rejecting ambiguous matches. RANSAC homography then
counts geometrically consistent **inliers**.

In [ ]:
# 2.3.1 — Reusable SIFT matching helpers
def ratio_filter(desA, desB, ratio=0.75):
    '''BF knn matching + Lowe ratio test -> list of good DMatch objects.'''
    bf = cv2.BFMatcher(cv2.NORM_L2, crossCheck=False)
    return [m for m, n in bf.knnMatch(desA, desB, k=2) if m.distance < ratio * n.distance]

def sift_match(grayA, kpA, desA, grayB, kpB, desB, ratio=0.75):
    '''Full pipeline: good matches + RANSAC homography; returns a stats dict.'''
    good = ratio_filter(desA, desB, ratio)
    inliers, M = 0, None
    if len(good) >= 4:
        src = np.float32([kpA[m.queryIdx].pt for m in good]).reshape(-1, 1, 2)
        dst = np.float32([kpB[m.trainIdx].pt for m in good]).reshape(-1, 1, 2)
        M, mask = cv2.findHomography(src, dst, cv2.RANSAC, 5.0)
        inliers = int(mask.ravel().sum()) if mask is not None else 0
    return {
        "keypoints_A": len(kpA), "keypoints_B": len(kpB),
        "good_matches": len(good), "ransac_inliers": inliers, "homography": M, "good_list": good,
    }

def draw_match_pair(imgA_color, kpA, imgB_gray, kpB, good, draw_n=50):
    '''Match visualisation between colour A (BGR) and a colourised copy of B (BGR).'''
    imgB_color = cv2.cvtColor(imgB_gray, cv2.COLOR_GRAY2BGR)
    return cv2.drawMatches(imgA_color, kpA, imgB_color, kpB, good[:draw_n], None,
                           flags=cv2.DRAW_MATCHES_FLAGS_NOT_DRAW_SINGLE_POINTS)

In [ ]:
# 2.3.2 — Match original vs second image
result_2way = sift_match(gray, kp1, des1, gray2, kp2, des2)
print(f"good (ratio) matches : {result_2way['good_matches']}")
print(f"RANSAC inliers       : {result_2way['ransac_inliers']}")

match_img_second = draw_match_pair(original_img, kp1, gray2, kp2, result_2way["good_list"], draw_n=50)
show_image(match_img_second, "SIFT BF matches - original vs second (top 50)", size=(13, 6))
save_output(2, "13_sift_bf_matching.png", match_img_second)

## 2.4 — Feature matching under rotation and resizing

The original is transformed four ways — 45 and 90 degree rotations (with expanding canvas),
50 % downscale and 150 % upscale — then matched **against its untransformed self**: a clean
invariance test, since every surviving match should be a correct one.

In [ ]:
# 2.4.1 — Generate transformed variants and match each to the original
def rotate_image(img, angle):
    '''Rotate around centre; canvas expands so corners are not clipped.'''
    hR, wR = img.shape[:2]
    Mrot = cv2.getRotationMatrix2D((wR / 2, hR / 2), angle, 1.0)
    cos, sin = abs(Mrot[0, 0]), abs(Mrot[0, 1])
    nW, nH = int(hR * sin + wR * cos), int(hR * cos + wR * sin)
    Mrot[0, 2] += (nW - wR) / 2
    Mrot[1, 2] += (nH - hR) / 2
    return cv2.warpAffine(img, Mrot, (nW, nH))

variants = {
    "rotated 45":   cv2.cvtColor(rotate_image(original_bgr, 45), cv2.COLOR_BGR2GRAY),
    "rotated 90":   cv2.cvtColor(rotate_image(original_bgr, 90), cv2.COLOR_BGR2GRAY),
    "resized 50%":  cv2.resize(gray, (W // 2, H // 2), interpolation=cv2.INTER_AREA),
    "resized 150%": cv2.resize(gray, (int(W * 1.5), int(H * 1.5)), interpolation=cv2.INTER_CUBIC),
}

siftV = cv2.SIFT_create()
invariance = {}
for name, gv in variants.items():
    kpv, desv = siftV.detectAndCompute(gv, None)
    invariance[name] = sift_match(gray, kp1, des1, gv, kpv, desv)
    s = invariance[name]
    print(f"{name:>13}: kps {len(kpv):>5} | good {s['good_matches']:>4} | inliers {s['ransac_inliers']:>4}")

In [ ]:
# 2.4.2 — Invariance results: bar chart
names = list(invariance.keys())
good_counts   = [invariance[n]["good_matches"] for n in names]
inlier_counts = [invariance[n]["ransac_inliers"] for n in names]

x = np.arange(len(names))
wBar = 0.35
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(x - wBar / 2, good_counts,   wBar, label="good (ratio) matches")
ax.bar(x + wBar / 2, inlier_counts, wBar, label="RANSAC inliers")
ax.set_xticks(x)
ax.set_xticklabels(names)
ax.set_title("SIFT matching invariance to rotation / resizing")
ax.legend()
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
out_dir2 = OUTPUT_DIR / ("Task " + str(2))
out_dir2.mkdir(parents=True, exist_ok=True)
fig.savefig(out_dir2 / "14_invariance_counts.png", bbox_inches="tight")
plt.show()

In [ ]:
# 2.4.3 — Visual check: matching the two hardest variants
invariance_vis = []
for name in ["rotated 45", "resized 50%"]:
    gv = variants[name]
    kpv, desv = siftV.detectAndCompute(gv, None)
    goodv = invariance[name]["good_list"]
    invariance_vis.append(draw_match_pair(original_img, kp1, gv, kpv, goodv, draw_n=40))

show_grid(
    [invariance_vis[0], invariance_vis[1]],
    ["original vs rotated 45", "original vs resized 50%"],
    cols=1, width_per=12,
    save_name="15_invariance_examples.png", task=2,
)

### Task 2 — Comparison of feature matching results

| Aspect | HOG | SIFT + Brute Force |
|---|---|---|
| Nature | dense descriptor over a grid | sparse local descriptors at keypoints |
| Output | one normalised feature vector per image/patch | keypoints, 128-D descriptors and geometry |
| Invariance | gradient-orientation statistics (no scale invariance) | scale + rotation, partially illumination |
| Typical use | sliding-window classification (persons, cars) | matching, panorama stitching, tracking |

**Invariance findings (quantified in the bar chart above):**

- **Rotation** — SIFT descriptors are computed relative to each keypoint's dominant
  orientation, so a large share of matches survives; matches degrade gradually with angle.
- **Downscale 50 %** — absolute keypoint count drops (fewer pixels, coarser Gaussian blur);
  surviving matches remain geometrically consistent (high inlier ratio).
- **Upscale 150 %** — more keypoints, but many appear only at the blurred/artefact scale
  levels, so the good-match count grows faster than it stays reliable.
- **RANSAC inliers** are the strongest quality indicator — they enforce a single global
  geometric transformation, not just descriptor similarity.

All figures are saved under `3. Outputs/Task 2/`.

## Task 3 — Face and Object Detection

Two complementary detectors are applied to the **original image** and to a **selected
frame of the video**:

| Detector | Family | Classes | Speed |
|---|---|---|---|
| **Viola-Jones** (Haar cascades) | hand-crafted features + boosted classifier | faces (frontal) | very fast, CPU-only |
| **YOLOv8-Nano** | single-shot deep CNN (pre-trained on COCO) | 80 object classes | real-time (GPU or CPU) 

## 3.1 — Viola-Jones face detection

The cascade scans a sliding window over an **integral image** and evaluates Haar-like
features; every stage of the trained Adaboot cascade rejects unlikely windows. Key tuning
parameters: `scaleFactor` (image pyramid step) and `minNeighbors` (merge threshold /
false-positive control).

In [ ]:
# 3.1.1 — Haar cascade face detection on the original image
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")
eye_cascade  = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_eye.xml")

faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(40, 40))
print(f"faces detected: {len(faces)}")

vjj_img = original_bgr.copy()
for (fx, fy, fw, fh) in faces:
    cv2.rectangle(vjj_img, (fx, fy), (fx + fw, fy + fh), (0, 255, 0), 2)
    cv2.putText(vjj_img, "face", (fx, max(20, fy - 8)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
    # optional detail: eyes inside each face region
    roi = gray[fy:fy + fh, fx:fx + fw]
    eyes = eye_cascade.detectMultiScale(roi, scaleFactor=1.1, minNeighbors=8)
    for (ex, ey, ew, eh) in eyes:
        cv2.rectangle(vjj_img, (fx + ex, fy + ey), (fx + ex + ew, fy + ey + eh), (0, 200, 255), 2)
if len(faces) == 0:
    print("(no frontal faces found with these parameters -> try minNeighbors=3 or scaleFactor=1.05)")

show_image(vjj_img, "Viola-Jones face detection (green = face, orange = eyes)", size=(7, 5.5))
save_output(3, "16_violajones_faces_image.png", vjj_img)

## 3.2 — Object detection with YOLOv8-Nano

`yolov8n.pt` (pre-trained on COCO, ~3.2M parameters) is the fastest YOLOv8 variant.
`ultralytics` auto-downloads the weights on first use. Boxes, labels and confidences are
drawn by the model's built-in `plot()` (returns a BGR array).

In [ ]:
# 3.2.1 — Load YOLOv8 Nano and detect on the original image
from ultralytics import YOLO

yolo = YOLO("yolov8n.pt")          # weights auto-download on first run
res_img = yolo.predict(str(ORIGINAL_IMG), conf=0.40, verbose=False)[0]

annotated_img = res_img.plot()      # BGR ndarray with boxes
print(f"objects detected in image: {len(res_img.boxes)}")
for box in res_img.boxes:
    cls_name = res_img.names[int(box.cls[0])]
    conf = float(box.conf[0])
    x1, y1, x2, y2 = [int(v) for v in box.xyxy[0]]
    print(f"  {cls_name:<12} conf={conf:.2f}  box=({x1},{y1})-({x2},{y2})")

show_image(annotated_img, "YOLOv8-Nano detection - original image", size=(7, 5.5))
save_output(3, "17_yolov8n_image.png", annotated_img)

## 3.3 — Detection on a selected video frame

The **middle frame** of the video is extracted and the same two detectors run on it:
Viola-Jones for faces, YOLOv8-Nano for COCO objects.

In [ ]:
# 3.3.1 — Extract middle video frame and run both detectors
cap = cv2.VideoCapture(str(VIDEO))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
mid_idx = max(0, total_frames // 2)
cap.set(cv2.CAP_PROP_POS_FRAMES, mid_idx)
ret, video_frame = cap.read()
cap.release()
assert ret, ("Could not decode frame %d from video" % mid_idx)
frame_gray = cv2.cvtColor(video_frame, cv2.COLOR_BGR2GRAY)
print(f"video frame selected: index {mid_idx} / {total_frames}")

# (a) Viola-Jones on the frame
faces_v = face_cascade.detectMultiScale(frame_gray, scaleFactor=1.1, minNeighbors=5, minSize=(40, 40))
vjj_frame = video_frame.copy()
for (fx, fy, fw, fh) in faces_v:
    cv2.rectangle(vjj_frame, (fx, fy), (fx + fw, fy + fh), (0, 255, 0), 2)
print(f"faces in video frame: {len(faces_v)}")

# (b) YOLOv8-Nano on the frame (ndarray input)
res_frame = yolo.predict(video_frame, conf=0.40, verbose=False)[0]
yolo_frame = res_frame.plot()
print(f"objects in video frame: {len(res_frame.boxes)}")
for box in res_frame.boxes:
    cls_name = res_frame.names[int(box.cls[0])]
    print(f"  {cls_name:<12} conf={float(box.conf[0]):.2f}")

show_grid(
    [vjj_frame, yolo_frame],
    [f"Viola-Jones on frame {mid_idx} ({len(faces_v)} faces)",
     f"YOLOv8-Nano on frame {mid_idx} ({len(res_frame.boxes)} objects)"],
    cols=2, width_per=5.2,
    save_name="18_detection_on_video_frame.png", task=3,
)

### Task 3 — Viola-Jones vs YOLOv8-Nano comparison

- **Viola-Jones** runs a hand-crafted feature cascade — extremely light and fast, but
  restricted to *frontal faces* and sensitive to pose, occlusion and lighting.
- **YOLOv8-Nano** is a compact deep detector generalised by COCO pre-training — it
  recognises 80 classes, is robust to pose/illumination, and gives per-object confidences,
  at the price of a model download and heavier compute.
- Practical observation recorded in the outputs: YOLO produces far more detections (all
  classes), while Haar cascades produce fewer, higher-precision face-local detections.

All figures are saved under `3. Outputs/Task 3/`.

## Task 4 — Video Analysis and Motion Tracking

Pipeline:

1. **Frame extraction** — decode the video and sample informative frames.
2. **Frame differencing** — motion between consecutive-ish frames via absolute difference.
3. **Background subtraction (MOG2)** — adaptive Gaussian-mixture background model.
4. **Lucas-Kanade optical flow** — sparse point tracking with Shi-Tomasi corners.
5. **Composite motion visualisation** of the mid-video moment.

In [ ]:
# 4.1.1 — Video properties + sampled frame extraction
cap = cv2.VideoCapture(str(VIDEO))
VID_FPS = cap.get(cv2.CAP_PROP_FPS)
TOTAL = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
VID_W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
VID_H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
print(f"video: {VID_W}x{VID_H} @ {VID_FPS:.2f} fps, {TOTAL} frames ({TOTAL / max(VID_FPS, 1e-9):.1f} s)")

def read_frame(cap, idx):
    '''Decode a specific frame index; returns ndarray or None.'''
    cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
    ret, fr = cap.read()
    return fr if ret else None

FRAC = [0.0, 0.25, 0.5, 0.75, max(0.0, 0.999)]
sample_idx = sorted(set(int(f * max(TOTAL - 1, 0)) for f in FRAC))
sample_frames = {i: read_frame(cap, i) for i in sample_idx}
sample_frames = {i: fr for i, fr in sample_frames.items() if fr is not None}
cap.release()

idx_first, idx_mid, idx_last = sample_idx[0], sample_idx[len(sample_idx) // 2], sample_idx[-1]
show_grid(
    [sample_frames[i] for i in sample_idx],
    ["frame %d" % i for i in sample_idx],
    cols=5, width_per=3.0,
    save_name="19_frame_extraction.png", task=4,
)
save_output(4, "20_frame_first.png", sample_frames[idx_first])
save_output(4, "21_frame_mid.png",   sample_frames[idx_mid])
save_output(4, "22_frame_last.png",  sample_frames[idx_last])

## 4.2 — Frame differencing

The absolute difference between two frames close in time highlights pixels that changed
— i.e. **moving objects**. A threshold separates real motion from sensor noise, and a
small morphological opening cleans residue.

In [ ]:
# 4.2.1 — Motion mask between the mid frame and a neighbour
DIFF_GAP = max(1, int(0.02 * TOTAL))          # frames between the compared pair
idxA = idx_mid
idxB = min(idxA + DIFF_GAP, TOTAL - 1)

cap_diff = cv2.VideoCapture(str(VIDEO))
frameA = read_frame(cap_diff, idxA)
frameB = read_frame(cap_diff, idxB)
cap_diff.release()

grayA = cv2.cvtColor(frameA, cv2.COLOR_BGR2GRAY)
grayB = cv2.cvtColor(frameB, cv2.COLOR_BGR2GRAY)

diff = cv2.absdiff(grayA, grayB)
_, motion_mask = cv2.threshold(diff, 30, 255, cv2.THRESH_BINARY)
motion_mask = cv2.morphologyEx(motion_mask, cv2.MORPH_OPEN, np.ones((3, 3), np.uint8))

motion_pct = np.count_nonzero(motion_mask) / motion_mask.size * 100
print(f"compared frames {idxA} and {idxB} (gap {DIFF_GAP}) -> {motion_pct:.2f}% pixels in motion")

out_dir4 = OUTPUT_DIR / ("Task " + str(4))
out_dir4.mkdir(parents=True, exist_ok=True)
fig, axes = plt.subplots(1, 5, figsize=(16, 3.2))
for ax, img, t in zip(axes,
                      [frameA, frameB, grayB, diff, motion_mask],
                      [f"frame {idxA}", f"frame {idxB}", "grayscale", "abs difference", "thresholded motion"]):
    show_image(img, title=t, ax=ax)
fig.tight_layout()
fig.savefig(out_dir4 / "23_frame_differencing.png", bbox_inches="tight")
plt.show()

save_output(4, "23a_diff_mask.png", motion_mask)

## 4.3 — Background subtraction with MOG2

`createBackgroundSubtractorMOG2` models each pixel's history with a Gaussian mixture and
classifies it as background (0), shadow (127) or foreground (255). Compared with frame
differencing it **adapts to lighting changes and static cameras**: the background model
improves as more frames are absorbed (`history=500`, `varThreshold=40`,
`detectShadows=True`).

In [ ]:
# 4.3.1 — MOG2 over a sampled pass through the video
MOG_MAX_FRAMES = min(TOTAL, 120)
MOG_STRIDE = max(1, TOTAL // MOG_MAX_FRAMES)

# sample-mask indices snapped onto the stride grid so they are actually processed
WATCH = sorted(set((round(f * max(TOTAL - 1, 0)) // MOG_STRIDE) * MOG_STRIDE for f in [0.25, 0.5, 0.75]))

mog2 = cv2.createBackgroundSubtractorMOG2(history=500, varThreshold=40, detectShadows=True)

mog_masks, mog_frames = {}, {}
processed = 0

cap = cv2.VideoCapture(str(VIDEO))
idx = 0
while processed < MOG_MAX_FRAMES:
    ret, fr = cap.read()
    if not ret:
        break
    if idx % MOG_STRIDE == 0:
        mask = mog2.apply(fr)
        if idx in WATCH:                    # keep masks/frames at the watched indices
            mog_masks[idx], mog_frames[idx] = mask, fr
        processed += 1
    idx += 1
cap.release()
print(f"MOG2 processed {processed} sampled frames; masks stored for indices {sorted(mog_masks)}")

In [ ]:
# 4.3.2 — Visualise MOG2 mask evolution (frame vs foreground mask)
mog_pairs, mog_titles = [], []
for i in sorted(mog_masks):
    mog_pairs += [mog_frames[i], mog_masks[i]]
    mog_titles += [f"frame {i}", f"MOG2 mask @ {i}"]

cols = 2
rows_m = math.ceil(len(mog_pairs) / cols)
fig, axes = plt.subplots(rows_m, cols, figsize=(10, 3.2 * rows_m))
axes = np.atleast_1d(axes).ravel()
for ax in axes:
    ax.axis("off")
for ax, img, t in zip(axes, mog_pairs, mog_titles):
    show_image(img, title=t, ax=ax)
fig.tight_layout()
fig.savefig(out_dir4 / "24_mog2_masks.png", bbox_inches="tight")
plt.show()

# clean + overlay one mask on its frame
mo = sorted(mog_masks)[-1]
clean = cv2.morphologyEx(mog_masks[mo], cv2.MORPH_OPEN, np.ones((3, 3), np.uint8))
clean = cv2.dilate(clean, np.ones((5, 5), np.uint8), iterations=1)
foreground_img = mog_frames[mo].copy()
foreground_img[clean == 255] = [0, 0, 255]             # moving pixels tinted red (RGB)
save_output(4, "25_mog2_overlay.png", foreground_img)

## 4.4 — Object tracking with Lucas-Kanade optical flow

Sparse LK tracking pipeline:

1. `goodFeaturesToTrack` (Shi-Tomasi corners) seeds points on the first frame.
2. `calcOpticalFlowPyrLK` follows them frame-to-frame with a patch-matching pyramid
   search (`winSize=21`, 3 pyramid levels).
3. Successful points (`status=1`) are chained into **tracks** drawn over the video.

In [ ]:
# 4.4.1 — LK sparse optical flow tracking
LK_FRAMES = min(TOTAL, 60)

lk_params = dict(
    winSize=(21, 21), maxLevel=3,
    criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 30, 0.01),
)

cap = cv2.VideoCapture(str(VIDEO))
ret, f0 = cap.read()
assert ret, "could not read first video frame"
gray_prev = cv2.cvtColor(f0, cv2.COLOR_BGR2GRAY)

p0 = cv2.goodFeaturesToTrack(gray_prev, maxCorners=150, qualityLevel=0.01, minDistance=10, blockSize=7)
if p0 is None or len(p0) < 3:
    print("too few trackable corners in first frame -> lowering qualityLevel")
    p0 = cv2.goodFeaturesToTrack(gray_prev, maxCorners=300, qualityLevel=0.003, minDistance=5)

track_overlay = np.zeros_like(f0)                      # black canvas accumulating trails
rng = np.random.default_rng(SEED)
point_colors = rng.integers(60, 256, (len(p0), 3)).tolist()

frames_tracked = 0
for step in range(LK_FRAMES - 1):
    ret, fr = cap.read()
    if not ret:
        break
    gray_next = cv2.cvtColor(fr, cv2.COLOR_BGR2GRAY)
    p1, st, err = cv2.calcOpticalFlowPyrLK(gray_prev, gray_next, p0, None, **lk_params)
    good_new = p1[st == 1]
    good_old = p0[st == 1]
    if len(good_new) < 3:
        print("tracking lost after", frames_tracked, "frames")
        break
    for k, ((x1, y1), (x0, y0)) in enumerate(zip(good_new, good_old)):
        c = point_colors[k % len(point_colors)]
        cv2.line(track_overlay, (int(x0), int(y0)), (int(x1), int(y1)), c, 2)
        cv2.circle(track_overlay, (int(x1), int(y1)), 2, c, -1)
    p0 = good_new.reshape(-1, 1, 2)
    gray_prev = gray_next
    frames_tracked += 1
cap.release()
print(f"LK tracked {len(p0)} surviving points across {frames_tracked} frames")

lk_result = cv2.addWeighted(f0, 0.75, track_overlay, 0.45, 0)
show_image(lk_result, "Lucas-Kanade optical flow tracks", size=(7, 5))
save_output(4, "26_lk_optical_flow.png", lk_result)

## 4.5 — Motion analysis overview (mid-video moment)

One combined figure gathers the four analyses for the mid-video moment:

In [ ]:
# 4.5.1 — Composite motion visualisation
mog_key = idx_mid if idx_mid in mog_masks else sorted(mog_masks)[-1]
mog_mid_side = mog_masks[mog_key]
diff_mid_mask = cv2.absdiff(
    cv2.cvtColor(sample_frames[idx_first], cv2.COLOR_BGR2GRAY),
    cv2.cvtColor(sample_frames[idx_mid], cv2.COLOR_BGR2GRAY),
)
_, diff_mid_mask = cv2.threshold(diff_mid_mask, 30, 255, cv2.THRESH_BINARY)

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
panels = [
    (sample_frames[idx_mid],      "sampled frame"),
    (diff_mid_mask,                        "frame difference (first vs mid)"),
    (mog_mid_side,                         "MOG2 foreground mask"),
    (track_overlay,               "LK optical flow trails"),
    (lk_result,                   "LK tracks over first frame"),
    (foreground_img,              "MOG2 foreground overlay"),
]
for ax, (img, t) in zip(axes.ravel(), panels):
    show_image(img, title=t, ax=ax)
fig.tight_layout()
fig.savefig(out_dir4 / "27_motion_overview.png", bbox_inches="tight")
plt.show()

### Task 4 — Results summary

- **Frame differencing** needs no model — cheapest motion signal, but sensitive to
  noise/frame gaps and cannot separate object from global illumination changes.
- **MOG2** builds an adaptive background and cleanly isolates foreground + shadows;
  performance depends on camera stability and the `varThreshold` sensitivity.
- **Lucas-Kanade** produces sparse but stable long-range tracks; points on moving
  objects accumulate visible trails, while static-background points stay fixed.
- The composite panel ties all analyses of the same moment together for the report.

All figures are saved under `3. Outputs/Task 4/`.

In [ ]:
# 6.1 — Outputs inventory (all saved figures per task)
for task in [1, 2, 3, 4]:
    d = OUTPUT_DIR / ("Task " + str(task))
    if d.is_dir():
        files = sorted(p.name for p in d.iterdir() if p.is_file())
        print(f"Task {task}: {len(files)} output file(s)")
        for f in files:
            print("   ", f)
    else:
        print(f"Task {task}: (no output folder yet)")

---
# Conclusion

This notebook implemented the complete CCA 3 practical assignment:

- **Task 1** demonstrated image degradation (Gaussian, salt & pepper), restoration
  (median / gaussian filtering) with quantitative **PSNR and SSIM** evaluation, colour
  quantisation with **KMeans**, and segmentation with **Otsu, morphology, Canny and
  watershed**.
- **Task 2** extracted **HOG** descriptors and **SIFT** keypoints, matched images with
  **Brute-Force + Lowe ratio + RANSAC**, and quantified robustness against **rotation
  and resizing**.
- **Task 3** compared classical **Viola-Jones** face detection with the deep
  **YOLOv8-Nano** detector on an image and a video frame.
- **Task 4** analysed video through **frame extraction, frame differencing,
  MOG2 background subtraction** and **Lucas-Kanade optical flow tracking**.

Together they show how classical image processing, feature-based methods and modern
deep learning complement each other across image, video, feature and detection problems.

*Detailed discussion: see the technical report in `4. Technical Report/`.*